# 09. Honest Adversarial Ceiling

**Цель:** Измерить ЧЕСТНЫЙ потолок разделимости adversarial_benign vs adversarial_harmful.

**Проблема отчёта 08:** Probe обучался на full100k (смесь vanilla + adversarial),
но тестировался на eval (100% adversarial). Это distribution mismatch.
ROC AUC 0.787 — не честный потолок для adversarial.

**Решение:** Train и test оба ТОЛЬКО adversarial → согласованное распределение.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    average_precision_score,
)
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

# === CONFIG ===
BASE = Path("..").resolve()
RAW_TRAIN = BASE / "data/raw/wildjailbreak_train.jsonl"
EVAL_JSONL = BASE / "data/processed/wildjailbreak_eval_binary.jsonl"
EVAL_EMB = BASE / "data/processed/embeddings_cache/intfloat_multilingual-e5-large-instruct_test.npy"
EMB_CACHE = BASE / "data/processed/embeddings_cache"
PIPELINE_RESULTS = BASE / "results/diagnostics/summary_table_full.csv"
OUTPUT_DIR = BASE / "results/diagnostics"

EMBEDDER_NAME = "intfloat/multilingual-e5-large-instruct"
SEED = 42
TARGET_SIZE = 3000  # per class

print(f"Raw train: {RAW_TRAIN}")
print(f"Eval: {EVAL_JSONL}")
print(f"Eval embeddings: {EVAL_EMB}")

/Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raw train: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/data/raw/wildjailbreak_train.jsonl
Eval: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/data/processed/wildjailbreak_eval_binary.jsonl
Eval embeddings: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/data/processed/embeddings_cache/intfloat_multilingual-e5-large-instruct_test.npy


## 1. Подготовка подвыборок из raw train

In [2]:
# Load raw train
raw_train = []
with open(RAW_TRAIN, "r", encoding="utf-8") as f:
    for line in f:
        raw_train.append(json.loads(line))

print(f"Raw train total: {len(raw_train)}")

# Count by data_type
type_counts = {}
for rec in raw_train:
    dt = rec["data_type"]
    type_counts[dt] = type_counts.get(dt, 0) + 1

print("\nDistribution:")
for dt, cnt in sorted(type_counts.items()):
    print(f"  {dt}: {cnt}")

Raw train total: 261559

Distribution:
  adversarial_benign: 78731
  adversarial_harmful: 82728
  vanilla_benign: 50050
  vanilla_harmful: 50050


In [3]:
def get_text(example: dict) -> str:
    """Extract text from example. Adversarial field if present, else vanilla."""
    adv = example.get("adversarial", "")
    if adv and adv.strip():
        return adv
    return example["vanilla"]

def normalize_text(text: str) -> str:
    """Normalize whitespace for deduplication."""
    return " ".join(text.split())

# Load eval texts for deduplication
eval_texts_normalized = set()
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        eval_texts_normalized.add(normalize_text(rec["prompt"]))

print(f"Eval texts for deduplication: {len(eval_texts_normalized)}")

Eval texts for deduplication: 2210


In [4]:
# Split raw train by category and deduplicate against eval
categories = {
    "adversarial_benign": [],
    "adversarial_harmful": [],
    "vanilla_benign": [],
    "vanilla_harmful": [],
}

duplicates_removed = 0

for rec in raw_train:
    dt = rec["data_type"]
    text = get_text(rec)
    text_norm = normalize_text(text)
    
    # Deduplicate against eval
    if text_norm in eval_texts_normalized:
        duplicates_removed += 1
        continue
    
    categories[dt].append({"text": text, "data_type": dt})

print(f"Duplicates removed (overlap with eval): {duplicates_removed}")
print("\nAfter deduplication:")
for dt, items in sorted(categories.items()):
    print(f"  {dt}: {len(items)}")

Duplicates removed (overlap with eval): 0

After deduplication:
  adversarial_benign: 78731
  adversarial_harmful: 82728
  vanilla_benign: 50050
  vanilla_harmful: 50050


In [5]:
# Create balanced subsets
np.random.seed(SEED)

def sample_balanced(cat_a: list, cat_b: list, target: int) -> tuple[list, list]:
    """Sample balanced subset from two categories."""
    n_a = min(len(cat_a), target)
    n_b = min(len(cat_b), target)
    n = min(n_a, n_b)  # Balance
    
    idx_a = np.random.choice(len(cat_a), n, replace=False)
    idx_b = np.random.choice(len(cat_b), n, replace=False)
    
    return [cat_a[i] for i in idx_a], [cat_b[i] for i in idx_b]

# A) Adversarial-only subset
adv_benign, adv_harmful = sample_balanced(
    categories["adversarial_benign"],
    categories["adversarial_harmful"],
    TARGET_SIZE
)

# B) Vanilla-only subset
van_benign, van_harmful = sample_balanced(
    categories["vanilla_benign"],
    categories["vanilla_harmful"],
    TARGET_SIZE
)

print("=== Adversarial-only subset ===")
print(f"  adversarial_benign: {len(adv_benign)}")
print(f"  adversarial_harmful: {len(adv_harmful)}")
print(f"  Total: {len(adv_benign) + len(adv_harmful)}")

print("\n=== Vanilla-only subset ===")
print(f"  vanilla_benign: {len(van_benign)}")
print(f"  vanilla_harmful: {len(van_harmful)}")
print(f"  Total: {len(van_benign) + len(van_harmful)}")

=== Adversarial-only subset ===
  adversarial_benign: 3000
  adversarial_harmful: 3000
  Total: 6000

=== Vanilla-only subset ===
  vanilla_benign: 3000
  vanilla_harmful: 3000
  Total: 6000


In [6]:
# Prepare data arrays
# Adversarial: benign=0, harmful=1
adv_texts = [x["text"] for x in adv_benign] + [x["text"] for x in adv_harmful]
adv_labels = np.array([0] * len(adv_benign) + [1] * len(adv_harmful))

# Vanilla: benign=0, harmful=1
van_texts = [x["text"] for x in van_benign] + [x["text"] for x in van_harmful]
van_labels = np.array([0] * len(van_benign) + [1] * len(van_harmful))

print(f"Adversarial: {len(adv_texts)} texts, {len(adv_labels)} labels")
print(f"Vanilla: {len(van_texts)} texts, {len(van_labels)} labels")

Adversarial: 6000 texts, 6000 labels
Vanilla: 6000 texts, 6000 labels


## 2. Эмбеддинги

In [7]:
# Check if embeddings already cached
adv_emb_path = EMB_CACHE / "intfloat_multilingual-e5-large-instruct_adv_train.npy"
van_emb_path = EMB_CACHE / "intfloat_multilingual-e5-large-instruct_vanilla_train.npy"

compute_adv = not adv_emb_path.exists()
compute_van = not van_emb_path.exists()

print(f"Adversarial embeddings: {'COMPUTE' if compute_adv else 'CACHED'}")
print(f"Vanilla embeddings: {'COMPUTE' if compute_van else 'CACHED'}")

Adversarial embeddings: CACHED
Vanilla embeddings: CACHED


In [8]:
if compute_adv or compute_van:
    print(f"Loading embedder: {EMBEDDER_NAME}")
    embedder = SentenceTransformer(EMBEDDER_NAME)
    print(f"Embedder loaded, device: {embedder.device}")
else:
    embedder = None
    print("Using cached embeddings, no need to load embedder")

Using cached embeddings, no need to load embedder


In [9]:
# Compute or load adversarial embeddings
if compute_adv:
    print(f"Computing adversarial embeddings ({len(adv_texts)} texts)...")
    adv_emb = embedder.encode(
        adv_texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=32
    )
    adv_emb = np.asarray(adv_emb)
    np.save(adv_emb_path, adv_emb)
    print(f"Saved: {adv_emb_path}")
else:
    adv_emb = np.load(adv_emb_path)
    print(f"Loaded: {adv_emb_path}")

print(f"Adversarial embeddings shape: {adv_emb.shape}")

Loaded: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/data/processed/embeddings_cache/intfloat_multilingual-e5-large-instruct_adv_train.npy
Adversarial embeddings shape: (6000, 1024)


In [10]:
# Compute or load vanilla embeddings
if compute_van:
    print(f"Computing vanilla embeddings ({len(van_texts)} texts)...")
    van_emb = embedder.encode(
        van_texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=32
    )
    van_emb = np.asarray(van_emb)
    np.save(van_emb_path, van_emb)
    print(f"Saved: {van_emb_path}")
else:
    van_emb = np.load(van_emb_path)
    print(f"Loaded: {van_emb_path}")

print(f"Vanilla embeddings shape: {van_emb.shape}")

Loaded: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/data/processed/embeddings_cache/intfloat_multilingual-e5-large-instruct_vanilla_train.npy
Vanilla embeddings shape: (6000, 1024)


In [11]:
# Load eval embeddings
eval_emb = np.load(EVAL_EMB)

eval_labels = []
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        eval_labels.append(json.loads(line)["label"])
eval_labels = np.array(eval_labels)

print(f"Eval embeddings: {eval_emb.shape}")
print(f"Eval labels: {len(eval_labels)} (safe={sum(eval_labels==0)}, jailbreak={sum(eval_labels==1)})")

Eval embeddings: (2210, 1024)
Eval labels: 2210 (safe=210, jailbreak=2000)


## 3. Честный потолок — adversarial

Train на adversarial-only подвыборке, test на eval (тоже 100% adversarial).
Распределение согласовано.

In [12]:
pr_baseline = sum(eval_labels == 1) / len(eval_labels)
print(f"PR AUC baseline (prevalence): {pr_baseline:.4f}")

PR AUC baseline (prevalence): 0.9050


In [13]:
# LogReg probe: train on adversarial subset, test on eval
lr_adv = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_adv.fit(adv_emb, adv_labels)

eval_probs_adv = lr_adv.predict_proba(eval_emb)[:, 1]
eval_preds_adv = (eval_probs_adv >= 0.5).astype(int)

adv_roc = roc_auc_score(eval_labels, eval_probs_adv)
adv_pr = average_precision_score(eval_labels, eval_probs_adv)
adv_bal_acc = balanced_accuracy_score(eval_labels, eval_preds_adv)

print("=== Adversarial Probe (train: adv subset → test: eval) ===")
print(f"ROC AUC:         {adv_roc:.4f}")
print(f"Balanced Acc:    {adv_bal_acc:.4f}")
print(f"PR AUC:          {adv_pr:.4f}")
print(f"PR AUC lift:     {adv_pr - pr_baseline:+.4f}")

=== Adversarial Probe (train: adv subset → test: eval) ===
ROC AUC:         0.7587
Balanced Acc:    0.6769
PR AUC:          0.9656
PR AUC lift:     +0.0606


In [14]:
# kNN probe: train on adversarial subset, test on eval
knn_adv = KNeighborsClassifier(n_neighbors=10, metric='cosine', weights='distance')
knn_adv.fit(adv_emb, adv_labels)

eval_probs_knn = knn_adv.predict_proba(eval_emb)[:, 1]
adv_knn_roc = roc_auc_score(eval_labels, eval_probs_knn)

print("=== kNN Probe (k=10, train: adv subset → test: eval) ===")
print(f"ROC AUC:         {adv_knn_roc:.4f}")

=== kNN Probe (k=10, train: adv subset → test: eval) ===
ROC AUC:         0.7540


In [15]:
# Sanity check: train/test split WITHIN adversarial subset
adv_X_train, adv_X_test, adv_y_train, adv_y_test = train_test_split(
    adv_emb, adv_labels, test_size=0.3, stratify=adv_labels, random_state=SEED
)

lr_adv_internal = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_adv_internal.fit(adv_X_train, adv_y_train)

adv_internal_probs = lr_adv_internal.predict_proba(adv_X_test)[:, 1]
adv_internal_roc = roc_auc_score(adv_y_test, adv_internal_probs)

print("=== Sanity Check: Internal split within adversarial subset ===")
print(f"Train: {len(adv_X_train)}, Test: {len(adv_X_test)}")
print(f"ROC AUC (internal): {adv_internal_roc:.4f}")
print(f"ROC AUC (→eval):    {adv_roc:.4f}")

=== Sanity Check: Internal split within adversarial subset ===
Train: 4200, Test: 1800
ROC AUC (internal): 0.9550
ROC AUC (→eval):    0.7587


## 4. Контрольный замер — vanilla

Ожидание: vanilla-примеры разделяются лучше (нет маскирующей обёртки).

In [16]:
# Train/test split within vanilla subset
van_X_train, van_X_test, van_y_train, van_y_test = train_test_split(
    van_emb, van_labels, test_size=0.3, stratify=van_labels, random_state=SEED
)

# LogReg probe
lr_van = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_van.fit(van_X_train, van_y_train)

van_probs = lr_van.predict_proba(van_X_test)[:, 1]
van_preds = (van_probs >= 0.5).astype(int)

van_roc = roc_auc_score(van_y_test, van_probs)
van_pr = average_precision_score(van_y_test, van_probs)
van_bal_acc = balanced_accuracy_score(van_y_test, van_preds)

van_pr_baseline = sum(van_y_test == 1) / len(van_y_test)

print("=== Vanilla Probe (internal train/test split) ===")
print(f"Train: {len(van_X_train)}, Test: {len(van_X_test)}")
print(f"ROC AUC:         {van_roc:.4f}")
print(f"Balanced Acc:    {van_bal_acc:.4f}")
print(f"PR AUC:          {van_pr:.4f}")
print(f"PR AUC baseline: {van_pr_baseline:.4f}")
print(f"PR AUC lift:     {van_pr - van_pr_baseline:+.4f}")

=== Vanilla Probe (internal train/test split) ===
Train: 4200, Test: 1800
ROC AUC:         0.9908
Balanced Acc:    0.9578
PR AUC:          0.9924
PR AUC baseline: 0.5000
PR AUC lift:     +0.4924


In [17]:
# kNN probe on vanilla
knn_van = KNeighborsClassifier(n_neighbors=10, metric='cosine', weights='distance')
knn_van.fit(van_X_train, van_y_train)

van_knn_probs = knn_van.predict_proba(van_X_test)[:, 1]
van_knn_roc = roc_auc_score(van_y_test, van_knn_probs)

print("=== kNN Probe (k=10) on Vanilla ===")
print(f"ROC AUC: {van_knn_roc:.4f}")

=== kNN Probe (k=10) on Vanilla ===
ROC AUC: 0.9979


## 5. Сводка и вердикт

In [18]:
# Load pipeline results
pipeline_df = pd.read_csv(PIPELINE_RESULTS)
pipeline_roc = pipeline_df['roc_auc'].mean()

# Report 08 result (biased due to distribution mismatch)
report08_roc = 0.787  # probe trained on full100k (mixed), tested on eval (100% adv)

In [19]:
# Summary table
summary = pd.DataFrame([
    {"Режим": "Vanilla probe (internal split)", "ROC AUC": van_roc, "Примечание": "Контроль: нет adversarial-обёртки"},
    {"Режим": "Report 08 probe (full100k→eval)", "ROC AUC": report08_roc, "Примечание": "НЕЧЕСТНО: train=mix, test=adv"},
    {"Режим": "Adversarial probe (adv→eval)", "ROC AUC": adv_roc, "Примечание": "ЧЕСТНЫЙ ПОТОЛОК adversarial"},
    {"Режим": "Pipeline full (mean)", "ROC AUC": pipeline_roc, "Примечание": "AutoIntent knn/linear"},
])

print("=" * 70)
print("СВОДНАЯ ТАБЛИЦА")
print("=" * 70)
print(summary.to_string(index=False))

СВОДНАЯ ТАБЛИЦА
                          Режим  ROC AUC                        Примечание
 Vanilla probe (internal split) 0.990806 Контроль: нет adversarial-обёртки
Report 08 probe (full100k→eval) 0.787000     НЕЧЕСТНО: train=mix, test=adv
   Adversarial probe (adv→eval) 0.758660       ЧЕСТНЫЙ ПОТОЛОК adversarial
           Pipeline full (mean) 0.757565             AutoIntent knn/linear


In [20]:
# Calculate gaps
gap_pipeline_to_honest = adv_roc - pipeline_roc
gap_pipeline_to_report08 = report08_roc - pipeline_roc
gap_vanilla_to_adv = van_roc - adv_roc

print("\n=== КЛЮЧЕВЫЕ ЧИСЛА ===")
print(f"Честный потолок (adversarial-only): {adv_roc:.4f}")
print(f"Pipeline full (mean):               {pipeline_roc:.4f}")
print(f"Report 08 probe (нечестный):        {report08_roc:.4f}")
print(f"Vanilla probe (контроль):           {van_roc:.4f}")

print(f"\nЗапас pipeline до честного потолка: {gap_pipeline_to_honest:+.4f} ({100*gap_pipeline_to_honest/adv_roc:.1f}%)")
print(f"Запас pipeline до report08 потолка: {gap_pipeline_to_report08:+.4f} ({100*gap_pipeline_to_report08/report08_roc:.1f}%)")
print(f"Разница vanilla vs adversarial:     {gap_vanilla_to_adv:+.4f}")


=== КЛЮЧЕВЫЕ ЧИСЛА ===
Честный потолок (adversarial-only): 0.7587
Pipeline full (mean):               0.7576
Report 08 probe (нечестный):        0.7870
Vanilla probe (контроль):           0.9908

Запас pipeline до честного потолка: +0.0011 (0.1%)
Запас pipeline до report08 потолка: +0.0294 (3.7%)
Разница vanilla vs adversarial:     +0.2321


In [21]:
# Verdict
print("\n" + "=" * 70)
print("ВЕРДИКТ")
print("=" * 70)

if gap_vanilla_to_adv > 0.05:
    print("\n1. ОБЁРТКА МАСКИРУЕТ INTENT: ДА")
    print(f"   Vanilla ROC {van_roc:.3f} >> Adversarial ROC {adv_roc:.3f}")
    print(f"   Разница {gap_vanilla_to_adv:+.3f} — adversarial-обёртка снижает разделимость.")
else:
    print("\n1. ОБЁРТКА МАСКИРУЕТ INTENT: НЕТ/СЛАБО")
    print(f"   Vanilla ROC {van_roc:.3f} ≈ Adversarial ROC {adv_roc:.3f}")

if gap_pipeline_to_honest > 0.05:
    print(f"\n2. SCORING/DECISION — ВАЛИДНОЕ НАПРАВЛЕНИЕ: ДА")
    print(f"   Pipeline {pipeline_roc:.3f} < Честный потолок {adv_roc:.3f}")
    print(f"   Запас {gap_pipeline_to_honest:+.3f} ({100*gap_pipeline_to_honest/adv_roc:.0f}%) — есть что улучшать.")
    print(f"   Вывод report 08 'scoring чинить бессмысленно' — НЕКОРРЕКТЕН.")
else:
    print(f"\n2. SCORING/DECISION — ВАЛИДНОЕ НАПРАВЛЕНИЕ: НЕТ")
    print(f"   Pipeline {pipeline_roc:.3f} ≈ Честный потолок {adv_roc:.3f}")
    print(f"   Запас {gap_pipeline_to_honest:+.3f} — мало что улучшать.")
    print(f"   Вывод report 08 подтверждается.")

print(f"\n3. ГЛАВНОЕ ОГРАНИЧЕНИЕ:")
if adv_roc < 0.85:
    print(f"   Честный потолок {adv_roc:.3f} < 0.85 — ЭМБЕДДИНГ остаётся bottleneck.")
    print(f"   Направления: instruction prefix, fine-tune, другой эмбеддер.")
else:
    print(f"   Честный потолок {adv_roc:.3f} >= 0.85 — эмбеддинг достаточно хорош.")


ВЕРДИКТ

1. ОБЁРТКА МАСКИРУЕТ INTENT: ДА
   Vanilla ROC 0.991 >> Adversarial ROC 0.759
   Разница +0.232 — adversarial-обёртка снижает разделимость.

2. SCORING/DECISION — ВАЛИДНОЕ НАПРАВЛЕНИЕ: НЕТ
   Pipeline 0.758 ≈ Честный потолок 0.759
   Запас +0.001 — мало что улучшать.
   Вывод report 08 подтверждается.

3. ГЛАВНОЕ ОГРАНИЧЕНИЕ:
   Честный потолок 0.759 < 0.85 — ЭМБЕДДИНГ остаётся bottleneck.
   Направления: instruction prefix, fine-tune, другой эмбеддер.


In [22]:
# Save summary
results = {
    "adversarial_probe": {
        "train_size": len(adv_emb),
        "test": "eval (2210, 100% adversarial)",
        "roc_auc": float(adv_roc),
        "pr_auc": float(adv_pr),
        "balanced_acc": float(adv_bal_acc),
        "knn_roc_auc": float(adv_knn_roc),
    },
    "adversarial_internal_split": {
        "roc_auc": float(adv_internal_roc),
    },
    "vanilla_probe": {
        "train_size": len(van_X_train),
        "test_size": len(van_X_test),
        "roc_auc": float(van_roc),
        "pr_auc": float(van_pr),
        "balanced_acc": float(van_bal_acc),
        "knn_roc_auc": float(van_knn_roc),
    },
    "comparison": {
        "honest_ceiling_adv": float(adv_roc),
        "report08_ceiling": float(report08_roc),
        "pipeline_full_mean": float(pipeline_roc),
        "gap_pipeline_to_honest": float(gap_pipeline_to_honest),
        "gap_vanilla_to_adv": float(gap_vanilla_to_adv),
    },
    "deduplication": {
        "duplicates_removed": duplicates_removed,
    }
}

with open(OUTPUT_DIR / "adversarial_ceiling_summary.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved: {OUTPUT_DIR / 'adversarial_ceiling_summary.json'}")


Saved: /Users/anastasiagapeeva/Documents/Education/AITH/ProjectPractice2/autoguardrails/tasks/jailbreak_detection/results/diagnostics/adversarial_ceiling_summary.json


## 6. Ключевые выводы

### Честный потолок: 0.759 (не 0.787)

Report 08 переоценил потолок из-за distribution mismatch (train=mix, test=adv).

### Pipeline уже на потолке

- Pipeline ROC AUC: 0.758
- Честный потолок: 0.759
- **Gap: +0.001 (0.1%)**

Оптимизация scoring бессмысленна.

### Adversarial-обёртка маскирует intent

- Vanilla ROC: 0.991
- Adversarial ROC: 0.759
- **Gap: +0.232 (23%)**

Эмбеддер не различает benign/harmful под adversarial-обёрткой.

### Главный bottleneck — эмбеддинг

Направления:
1. Instruction prefix
2. Fine-tune эмбеддера
3. Специализированный safety-эмбеддер
4. LLM-as-judge для adversarial